# DST-HGNN (Kaggle GPU optimized)

**Dynamic Spatio-Temporal Graph Neural Networks for Explainable Health Prediction**

**Notebook purpose:** Full implementation (graph + text fusion) optimized for Kaggle GPU. This notebook includes:
- Environment setup (install commands for Kaggle)
- Data loading & weak/pseudo-labeling
- Text embedding (SentenceTransformers / transformers)
- Graph construction: patient temporal chains + semantic kNN edges
- DST-HGNN model (PyTorch + PyG skeleton)
- Training loop with mixed precision (AMP)
- Evaluation: confusion matrix, ROC-AUC, learning curves, t-SNE
- XAI: SHAP for baseline, Captum (Integrated Gradients) for GNN

**Notes:** Run this notebook on a Kaggle GPU runtime. Some install cells will need to be executed on Kaggle (internet required for pip installs).


In [ ]:
# === Setup: run on Kaggle GPU (execute this cell first) ===
# Note: Kaggle has internet during notebook runs; these installs are typical.
# If a command fails, please follow Kaggle-specific PyG install instructions.

# Core libraries
!pip install -q sentence-transformers==2.2.2 transformers==4.35.0 scikit-learn==1.3.2 xgboost==1.7.6 shap==0.41.0 captum==0.6.0

# PyTorch & PyG: Kaggle often has torch preinstalled; install matching torch_geometric
# For PyG, see https://pytorch-geometric.readthedocs.io/en/latest/notes/installation.html
!pip install -q torch_scatter torch_sparse torch_cluster torch_spline_conv torch_geometric -f https://data.pyg.org/whl/torch-2.2.0+cu118.html

# Helpful utilities
!pip install -q tqdm umap-learn==0.5.4

print('Install commands finished. Restart the runtime if necessary.')


In [ ]:
# === Imports & device ===
import os
import re
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Torch & device
import torch
print('torch version:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Optional: set seed
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


In [ ]:
# === Load dataset ===
# Adjust csv_path to where you've uploaded the dataset on Kaggle (or keep /mnt/data for local runs)
csv_path = '/kaggle/working/TCGA_Reports.csv'
if not os.path.exists(csv_path):
    csv_path = '/mnt/data/TCGA_Reports.csv'

print('Using csv_path =', csv_path)
df = pd.read_csv(csv_path, low_memory=False)
print('shape:', df.shape)
print(df.columns.tolist())

df.head()


In [ ]:
# === Weak / Pseudo-labeling ===
import re

keywords_map = {
    r'carcinoma|adenocarcinoma|malignant|cancer': 'malignant',
    r'benign|non[- ]neoplastic|no evidence of malignancy': 'benign',
    r'metastasis|metastatic': 'metastatic'
}

def label_from_filename(fname):
    if not isinstance(fname, str):
        return None
    f = fname.lower()
    if 'benign' in f:
        return 'benign'
    if 'malign' in f or 'cancer' in f or 'tumor' in f or 'carcinoma' in f:
        return 'malignant'
    if 'metast' in f:
        return 'metastatic'
    return None

def label_from_text(text):
    if not isinstance(text, str):
        return None
    t = text.lower()
    for pat, lab in keywords_map.items():
        if re.search(pat, t):
            return lab
    return None

if 'patient_filename' in df.columns:
    df['lbl_fname'] = df['patient_filename'].apply(label_from_filename)
else:
    df['lbl_fname'] = None

if 'text' in df.columns:
    df['lbl_text'] = df['text'].apply(label_from_text)
else:
    df['lbl_text'] = None

def combine_labels(row):
    if row['lbl_fname']:
        return row['lbl_fname']
    if row['lbl_text']:
        return row['lbl_text']
    return 'unknown'

df['weak_label'] = df.apply(combine_labels, axis=1)
print(df['weak_label'].value_counts())


In [ ]:
# === Text embeddings (SentenceTransformers) ===
from sentence_transformers import SentenceTransformer
MODEL_NAME = 'all-MiniLM-L6-v2'
print('Loading SBERT model', MODEL_NAME)
embedder = SentenceTransformer(MODEL_NAME, device=device)

texts = df['text'].fillna('').astype(str).tolist()
BATCH_SIZE = 64
embs = []
for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i:i+BATCH_SIZE]
    e = embedder.encode(batch, show_progress_bar=False, convert_to_numpy=True)
    embs.append(e)
embeddings = np.vstack(embs)
print('embeddings shape:', embeddings.shape)

import joblib
joblib.dump(embeddings, '/kaggle/working/embeddings.npy')
print('Saved embeddings to /kaggle/working/embeddings.npy')


In [ ]:
# === Cluster unknowns for pseudo-labels ===
from sklearn.cluster import KMeans
unknown_idx = df[df['weak_label']=='unknown'].index.tolist()
print('unknown count:', len(unknown_idx))

if len(unknown_idx) > 0:
    k = 3
    km = KMeans(n_clusters=k, random_state=SEED)
    km_labels = km.fit_predict(embeddings[unknown_idx])
    for idx, cl in zip(unknown_idx, km_labels):
        df.loc[idx, 'weak_label'] = f'cluster_{cl}'

print('Label distribution after clustering:')
print(df['weak_label'].value_counts())


In [ ]:
# === Train/Test split ===
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['weak_label'])

class_counts = df['label_enc'].value_counts()
valid_classes = class_counts[class_counts >= 10].index.tolist()
mask = df['label_enc'].isin(valid_classes)
df = df[mask].reset_index(drop=True)

# reload embeddings for filtered df
import joblib
embeddings = joblib.load('/kaggle/working/embeddings.npy')
embeddings = embeddings[mask.values]

X = embeddings
y = df['label_enc'].values

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(X, y, df.index.values, test_size=0.2, stratify=y, random_state=SEED)
print('train/test sizes:', X_train.shape, X_test.shape)


In [ ]:
# === Graph construction ===
from sklearn.neighbors import NearestNeighbors
import torch

n_nodes = len(df)
node_idx = {idx:i for i, idx in enumerate(df.index.tolist())}
edge_list = []

# temporal chains
if 'patient_filename' in df.columns:
    def patient_id(fname):
        if not isinstance(fname, str):
            return None
        return re.split('[_\-]', fname)[0]
    df['patient_id'] = df['patient_filename'].apply(patient_id)
    for pid, group in df.groupby('patient_id'):
        if pd.isnull(pid):
            continue
        ids = group.index.tolist()
        for a, b in zip(ids[:-1], ids[1:]):
            edge_list.append((node_idx[a], node_idx[b]))
            edge_list.append((node_idx[b], node_idx[a]))

# semantic kNN
K = 5
nbrs = NearestNeighbors(n_neighbors=K+1, algorithm='auto').fit(embeddings)
knn_dist, knn_idx = nbrs.kneighbors(embeddings)
for i in range(len(embeddings)):
    for j in knn_idx[i][1:]:
        edge_list.append((i, j))

edge_set = set(edge_list)
edge_index = torch.tensor(list(edge_set), dtype=torch.long).t().contiguous()
print('edge_index shape:', edge_index.shape)

x = torch.tensor(embeddings, dtype=torch.float)
y = torch.tensor(df['label_enc'].values, dtype=torch.long)

# timestamps
timestamps = torch.zeros((n_nodes, 1), dtype=torch.float)
if 'patient_id' in df.columns:
    for pid, group in df.groupby('patient_id'):
        ids = group.index.tolist()
        for t, idx in enumerate(ids):
            timestamps[node_idx[idx], 0] = t

print('x shape, y shape, timestamps shape:', x.shape, y.shape, timestamps.shape)


In [ ]:
# === DST-HGNN model (PyG) ===
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.data import Data

class TemporalGNN(torch.nn.Module):
    def __init__(self, in_dim, hid_dim, out_dim, heads=4):
        super().__init__()
        self.time_enc = torch.nn.Linear(1, in_dim)
        self.gat1 = GATConv(in_dim, hid_dim, heads=heads, concat=True)
        self.gat2 = GATConv(hid_dim*heads, out_dim, heads=1, concat=False)

    def forward(self, x, edge_index, timestamps=None):
        if timestamps is not None:
            x = x + self.time_enc(timestamps)
        x = F.elu(self.gat1(x, edge_index))
        x = self.gat2(x, edge_index)
        return x

from torch_geometric.data import Data
data = Data(x=x, edge_index=edge_index, y=y)

# train/test masks
train_mask = torch.zeros(n_nodes, dtype=torch.bool)
test_mask = torch.zeros(n_nodes, dtype=torch.bool)
for i in idx_train:
    train_mask[node_idx[i]] = True
for i in idx_test:
    test_mask[node_idx[i]] = True

data.train_mask = train_mask
data.test_mask = test_mask

IN_DIM = x.shape[1]
HID = 64
OUT_DIM = len(df['label_enc'].unique())
model = TemporalGNN(IN_DIM, HID, OUT_DIM).to(device)
print(model)


In [ ]:
# === Training loop (transductive) ===
from torch.optim import AdamW
from sklearn.metrics import f1_score

optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scaler = torch.cuda.amp.GradScaler()

EPOCHS = 30
history = {'train_loss':[], 'train_f1':[], 'test_f1':[]}

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    with torch.cuda.amp.autocast(enabled=(device.type=='cuda')):
        out = model(data.x.to(device), data.edge_index.to(device), timestamps.to(device))
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask].to(device))
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    # evaluate
    model.eval()
    with torch.no_grad():
        logits = model(data.x.to(device), data.edge_index.to(device), timestamps.to(device)).cpu()
    tr_preds = logits[data.train_mask].argmax(dim=1).numpy()
    tr_true = data.y[data.train_mask].numpy()
    te_preds = logits[data.test_mask].argmax(dim=1).numpy()
    te_true = data.y[data.test_mask].numpy()
    tr_f1 = f1_score(tr_true, tr_preds, average='macro')
    te_f1 = f1_score(te_true, te_preds, average='macro')
    history['train_loss'].append(loss.item())
    history['train_f1'].append(tr_f1)
    history['test_f1'].append(te_f1)
    print(f'Epoch {epoch+1}/{EPOCHS} loss={loss.item():.4f} tr_f1={tr_f1:.4f} te_f1={te_f1:.4f}')

# save
os.makedirs('/kaggle/working', exist_ok=True)
torch.save(model.state_dict(), '/kaggle/working/dst_hgnn_model.pth')
print('Saved model to /kaggle/working/dst_hgnn_model.pth')


In [ ]:
# === Evaluation & Visualization ===
model.eval()
with torch.no_grad():
    logits = model(data.x.to(device), data.edge_index.to(device), timestamps.to(device)).cpu()
probs = torch.softmax(logits, dim=1).numpy()
preds = probs.argmax(axis=1)

# Confusion matrix for test nodes
from sklearn.metrics import confusion_matrix
te_true = data.y[data.test_mask].numpy()
te_pred = preds[data.test_mask.numpy()]
cm = confusion_matrix(te_true, te_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d')
plt.title('Confusion matrix (test)')
plt.show()

# ROC curves
from sklearn.preprocessing import label_binarize
n_classes = OUT_DIM
true_bin = label_binarize(te_true, classes=list(range(n_classes)))
probs_test = probs[data.test_mask.numpy()]

from sklearn.metrics import roc_curve, auc
fpr = dict(); tpr = dict(); roc_auc = dict()
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(true_bin[:, i], probs_test[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(8,6))
for i in range(n_classes):
    plt.plot(fpr[i], tpr[i], label=f'class {i} (AUC = {roc_auc[i]:.2f})')
plt.plot([0,1],[0,1],'k--')
plt.legend(); plt.title('ROC Curves')
plt.show()

# Learning curves
plt.figure(); plt.plot(history['train_f1'], label='train_f1'); plt.plot(history['test_f1'], label='test_f1'); plt.legend(); plt.title('F1 over epochs'); plt.show()

# UMAP/t-SNE visualization
from umap import UMAP
umap = UMAP(n_components=2, random_state=SEED)
proj = umap.fit_transform(embeddings)
plt.figure(figsize=(8,6))
plt.scatter(proj[data.test_mask.numpy(),0], proj[data.test_mask.numpy(),1], c=te_true, cmap='tab10', s=6)
plt.title('UMAP of embeddings (test nodes)')
plt.show()


In [ ]:
# SHAP for baseline XGBoost
import xgboost as xgb
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

pipe = make_pipeline(TfidfVectorizer(max_features=20000), xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss'))
texts = df['text'].fillna('').tolist()
X_train_text = [texts[i] for i in idx_train]
X_test_text = [texts[i] for i in idx_test]
pipe.fit(X_train_text, data.y.numpy()[idx_train])

import shap
explainer = shap.Explainer(pipe.named_steps['xgb'])
sample = pipe.named_steps['tfidf'].transform(X_test_text[:100])
shap_values = explainer(sample)
shap.summary_plot(shap_values, feature_names=pipe.named_steps['tfidf'].get_feature_names_out())


In [ ]:
# Captum Integrated Gradients for GNN (sketch)
from captum.attr import IntegratedGradients

def model_forward(x_input):
    model.eval()
    with torch.no_grad():
        out = model(x_input.to(device), data.edge_index.to(device), timestamps.to(device))
    return out

ig = IntegratedGradients(model_forward)
# select a test node
test_nodes = [i for i in range(n_nodes) if data.test_mask[i]]
if len(test_nodes)>0:
    node_to_explain = test_nodes[0]
    attr = ig.attribute(inputs=data.x[node_to_explain].unsqueeze(0).to(device), target=int(data.y[node_to_explain].item()))
    print('Attribution shape:', attr.shape)


Notebook generated at `/mnt/data/DST_HGNN_Kaggle_GPU.ipynb`. Run on Kaggle GPU; execute the setup cell first.